# Knowledge Graph Generation Pipeline with Relation/Entity Hints (KG-Gen)

- **Setup & data**: locates the project root/`DATA` folder, selects a category and an LLM, loads the article subset, and also loads pre-extracted **relation hints** and **entity hints** (JSON files) used to guide extraction.
- **Per-article extraction**: runs `KGGen.generate` on every article in parallel (custom rate limiter + capped `ThreadPoolExecutor`), passing the relevant relation/entity hints for each article, and saves one graph pickle per article with automatic retry on failure.
- **Aggregation & clustering**: merges all per-article graphs into one combined graph, then deduplicates/clusters entities and relations (LLM-based) into the final `clustered_graph`, exported as pickle/JSON/HTML.
- **Provenance tracking**: propagates article-level provenance from the raw triplets through aggregation and clustering, so every final triplet keeps track of which source article(s) it came from.
- **Sanity checks**: verifies that every clustered triplet has a provenance entry, and cross-checks the number of triplets per article against that article's word count.

In [ ]:
from pathlib import Path
ROOT = Path().resolve()
while not (ROOT / "DATA").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
DATA = ROOT / "DATA"
print("CLEAN root:", ROOT)


In [ ]:
import re
import pandas as pd
import bs4
import requests
import networkx as nx
import json

import matplotlib.pyplot as plt
from tqdm import tqdm

pd.set_option('display.max_colwidth', 200)
%matplotlib inline

In [ ]:
from pathlib import Path


# ============ CATEGORY SELECTION ============
CATEGORY = "artesania"   # <-- just change this (danza, literatura, religion, ...)
MODELE : str = "mistral/mistral-small-2506"  # <-- just change this (mistral/mistral-small-2506, mistral/mistral-7b-instruct-v0.1, ...)
MODELE_title = MODELE.split("/")[-1].replace("-", "_")  # used to name output files

# ============ PATHS ============
SUBSETS_DIR = DATA / "SUBSETS_with_relations" / "ARTICLES_SUBSETS_ES"
RESULTS_BASE = DATA / "SUBSETS_with_relations" / "GRAPHS"
# input file for the selected category
input_file = SUBSETS_DIR / f"{CATEGORY}_articles_es_disjoint.csv"
if not input_file.exists():
    raise FileNotFoundError(f"Dataset not found: {input_file}")

# ============ FOLDER STRUCTURE ============
# RESULTS/GRAPHS/KG-GEN/SUBSETS/<category>/
#   ├── graphs_per_article/         <- one graph per article (.pkl)
#   ├── aggregate_graph.html/json   <- aggregated graph (next to the folder)
#   └── clustered_graph.html/json   <- clustered graph (next to the folder)

CATEGORY_DIR = RESULTS_BASE / CATEGORY 
PER_ARTICLE_DIR = CATEGORY_DIR / "graphs_per_article"
CLUSTERED_DIR = CATEGORY_DIR / "clustered_graph"
COMBINED_DIR = CATEGORY_DIR / "combined_graph"
PROVENANCE_DIR = CATEGORY_DIR / "provenance"

# create folders
PER_ARTICLE_DIR.mkdir(parents=True, exist_ok=True)
CLUSTERED_DIR.mkdir(parents=True, exist_ok=True)
COMBINED_DIR.mkdir(parents=True, exist_ok=True)
PROVENANCE_DIR.mkdir(parents=True, exist_ok=True)


# ============ LOADING EXISTING RELATIONS (if any) ============
existing_relations_file = DATA / "SUBSETS_with_relations" /"RELATIONS_EXTRAITES" / f"relations_extraites_{CATEGORY}_es.json"
dict_existing_relations = json.load(open(existing_relations_file, "r", encoding="utf-8"))

# ============ LOADING EXTRACTED ENTITIES ============
existing_entities_file = DATA / "SUBSETS_with_relations" /"ENTITES_EXTRAITES" / f"entites_extraites_{CATEGORY}_es.json"
dict_existing_entities = json.load(open(existing_entities_file, "r", encoding="utf-8"))

# kept for compatibility with the rest of the existing code

# ============ DATA LOADING ============
df_category = pd.read_csv(input_file)
full_text = " ".join(df_category["content"].dropna().astype(str))

print(f"📂 Category           : {CATEGORY}")
print(f"📄 Input file         : {input_file}")
print(f"📁 Category folder    : {CATEGORY_DIR}")
print(f"📁 Per-article graphs : {PER_ARTICLE_DIR}")
print(f"─" * 60)
print(f"Number of articles    : {len(df_category)}")
print(f"Number of characters  : {len(full_text):,}")
print(f"Number of words (approx.): {len(full_text.split()):,}")


print(f"Number of articles with suggested relations: {len(dict_existing_relations)}")
print(f"Number of articles with suggested entities: {len(dict_existing_entities)}")

# ==== API KEY ============
API_KEY= ""

In [ ]:
import subprocess
import os

# 1. use a dynamic path suitable for Linux/your machine
home_dir = os.path.expanduser("~")
target_path = os.path.join(home_dir, "LLM_BIAS_UCHILE/kg-gen/src/")

try:
    # 2. always add a timeout (in seconds)
    result = subprocess.run(
        ["grep", "-rn", "--exclude-dir=.git", "DeduplicationResult", target_path], 
        capture_output=True, 
        text=True,
        timeout=10  # the script will stop after 10s max instead of hanging
    )
    output = result.stdout
except subprocess.TimeoutExpired:
    print("Error: the grep search timed out.")
    output = ""
except FileNotFoundError:
    print("Error: folder or grep command not found.")
    output = ""

In [ ]:
import litellm
from kg_gen import KGGen
import os
import pickle
from pathlib import Path
from tqdm import tqdm  # progress bar
from sentence_transformers import SentenceTransformer


# Config
litellm.num_retries = 5

kg = KGGen(
    model=MODELE,
    api_key=API_KEY,
)

CHUNK_SIZE = 5000
graphs = []
kg.retrieval_model = SentenceTransformer("all-MiniLM-L6-v2")


from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

# lock to protect shared prints/lists
_lock = threading.Lock()
graphs = {}
index_to_article_id = {}
failed = []

def process_article(i, row):
    content = str(row["content"])
    title = row["title"]
    article_id = row["article_id"]
    # retrieve the precomputed relation/entity hints for this article
    relation_hints = dict_existing_relations.get(str(article_id), []) or []
    entity_hints = dict_existing_entities.get(str(article_id), []) or []
    output_file = PER_ARTICLE_DIR / f"graph_{i}_{title.replace(' ', '_')[:20]}.pkl"
    # print(relation_hints)
    # print(entity_hints)
    if output_file.exists():
        with open(output_file, "rb") as f:
            g = pickle.load(f)
        with _lock:
            print(f"  ✅ [{i}] {title} — already processed")
        return ("ok", i, title, g, article_id)

    try:
        g = kg.generate(
            input_data=content,
            chunk_size=CHUNK_SIZE,
            relation_hints=relation_hints if relation_hints else None,
            entity_hints=entity_hints if entity_hints else None,
        )
        with open(output_file, "wb") as f:
            pickle.dump(g, f)
        with _lock:
            print(f"  ✅ [{i}] {title} — saved")
        return ("ok", i, title, g, article_id)
    except Exception as e:
        with _lock:
            print(f"  ❌ [{i}] {title}: {e}")
        return ("fail", i, title, str(e), article_id)


# ⚙️ Parallelism tuning
# if kg.generate already uses N internal workers for chunks,
# and you want ~6 req/s in total, aim for N_outer * N_inner ≈ 6-10
N_OUTER = 30   # nb of articles in parallel

with ThreadPoolExecutor(max_workers=N_OUTER) as ex:
    futures = [
        ex.submit(process_article, i, row)
        for i, row in df_category.reset_index(drop=True).iterrows()
    ]
    for fut in tqdm(as_completed(futures), total=len(futures)):
        status, i, title, payload, article_id = fut.result()
        if status == "ok":
            graphs[i] = payload
            index_to_article_id[i] = article_id
        else:
            failed.append((i, title, payload, article_id))

print(f"\n✅ Graphs generated: {len(graphs)} / {len(df_category)}")
print(f"❌ Failures: {len(failed)}")

In [ ]:
# Retry only the failed articles
retry_failed = []
recovered = []

for i, title, err, article_id in failed:
    row = df_category.reset_index(drop=True).iloc[i]
    relation_hints = dict_existing_relations.get(str(article_id), []) or []
    entity_hints = dict_existing_entities.get(str(article_id), []) or []
    output_file = PER_ARTICLE_DIR / f"graph_{i}_{title.replace(' ', '_')[:20]}.pkl"
    
    # safety check: might have been saved in the meantime
    if output_file.exists():
        with open(output_file, "rb") as f:
            graphs.append(pickle.load(f))
        print(f"  ⏭️  [{i}] {title} already saved, skip")
        continue
    
    try:
        print(f"  🔄 [{i}] {title} — retry...")
        g = kg.generate(
            input_data=str(row["content"]),
            chunk_size=1000,  # smaller chunk size on retry
            relation_hints=relation_hints if relation_hints else None,
            entity_hints=entity_hints if entity_hints else None,
        )
        with open(output_file, "wb") as f:
            pickle.dump(g, f)
        graphs[i] = g
        recovered.append((i, title))
        index_to_article_id[i] = article_id
        print(f"  ✅ [{i}] {title} recovered!")
    except Exception as e:
        print(f"  ❌ [{i}] {title} failed again: {str(e)[:120]}")
        retry_failed.append((i, title, str(e)))

print(f"\n🎉 Recovered: {len(recovered)} / {len(failed)}")
print(f"💀 Definitively KO: {len(retry_failed)}")


In [ ]:

combined_graph = kg.aggregate(list(graphs.values()))  # merge all per-article graphs into one
print(f"\n📊 Aggregated graph:")
print(f"  - entities: {len(combined_graph.entities)}")
print(f"  - edges (relation types): {len(combined_graph.edges)}")
print(f"  - relations (triplets): {len(combined_graph.relations)}")

KGGen.visualize(combined_graph, output_path=str(CATEGORY_DIR / f"{CATEGORY}_aggregate_graph.html"),
    open_in_browser=False)

# PICKLE DUMP COMBINED GRAPH
combined_graph_file = COMBINED_DIR / f"{CATEGORY}_combined_graph.pkl"
COMBINED_DIR.mkdir(parents=True, exist_ok=True)
with open(combined_graph_file, "wb") as f:
    pickle.dump(combined_graph, f)

In [ ]:
import time
import threading
import litellm

# ============ RESET if already patched ============
if getattr(litellm.completion, "_is_rate_limited", False):
    if hasattr(litellm.completion, "_original"):
        litellm.completion = litellm.completion._original
    print("♻️  Old wrapper removed")

_original_completion = litellm.completion

# ============ SHARED STATE ============
_lock = threading.Lock()
_last_call_time = [0.0]
_call_count = [0]
_fail_count = [0]
_start_time = [time.time()]

MIN_INTERVAL = 0.20    # effective 5 req/s (margin under Mistral's 6/s limit)
TRUNCATE = 150

# ============ WRAPPER ============
def rate_limited_completion(*args, **kwargs):
    # 🔒 GLOBAL serialization: the sleep happens INSIDE the lock
    with _lock:
        elapsed = time.time() - _last_call_time[0]
        wait = MIN_INTERVAL - elapsed
        if wait > 0:
            time.sleep(wait)
        _last_call_time[0] = time.time()
        _call_count[0] += 1
        n = _call_count[0]
        total_elapsed = time.time() - _start_time[0]
        rate = n / total_elapsed if total_elapsed > 0 else 0.0
        fails = _fail_count[0]

    # compact log (otherwise 1500 calls = unreadable)
    if n % 25 == 0 or n <= 5:
        print(f"📞 #{n:4d} | t+{total_elapsed:6.1f}s | {rate:.2f}/s | fails={fails}")

    t0 = time.time()
    try:
        result = _original_completion(*args, **kwargs)
        return result
    except Exception as e:
        with _lock:
            _fail_count[0] += 1
        dt = time.time() - t0
        # only display real issues
        if "RateLimit" in type(e).__name__ or "429" in str(e):
            print(f"   ⚠️  #{n} 429 in {dt:.1f}s → DSPy retry")
        else:
            print(f"   ❌ #{n} {type(e).__name__} in {dt:.1f}s: {str(e)[:100]}")
        raise

rate_limited_completion._is_rate_limited = True
rate_limited_completion._original = _original_completion
litellm.completion = rate_limited_completion

print(f"✅ Rate limiter installed: {1/MIN_INTERVAL:.1f} req/s max (Mistral limit=6/s)")

In [ ]:
import concurrent.futures

_OriginalTPE = concurrent.futures.ThreadPoolExecutor

class CappedTPE(_OriginalTPE):
    def __init__(self, max_workers=None, *args, **kwargs):
        capped = min(max_workers or 5, 5)  # never exceed 5 workers, regardless of what's requested
        super().__init__(max_workers=capped, *args, **kwargs)

concurrent.futures.ThreadPoolExecutor = CappedTPE
print("✅ ThreadPoolExecutor capped at 6 workers max")

In [ ]:
from sentence_transformers import SentenceTransformer


# ============ STEP 3: clustering ============

kg_cluster=  KGGen(
    model=MODELE,
    api_key=API_KEY,
)

from kg_gen.steps._3_deduplicate import DeduplicateMethod

kg_cluster.retrieval_model = SentenceTransformer("all-MiniLM-L6-v2")

clustered_graph = kg_cluster.deduplicate(
    combined_graph,
    method=DeduplicateMethod.LM_BASED,  # or .SEMHASH for a fast, LLM-free option
)

print(f"\n🔗 After clustering:")
print(f"  - entities: {len(clustered_graph.entities)}")
print(f"  - edges: {len(clustered_graph.edges)}")
print(f"  - relations: {len(clustered_graph.relations)}")
print(f"  - entity clusters: {len(clustered_graph.entity_clusters)}")
print(f"  - edge clusters: {len(clustered_graph.edge_clusters)}")


# SAVE THE CLUSTERED GRAPH:
with open(CATEGORY_DIR / f"{CATEGORY}_clustered_graph.pkl", "wb") as f:
    pickle.dump(clustered_graph, f)
print(f"\n✅ Clustered graph saved at: {CATEGORY_DIR / f'{CATEGORY}_clustered_graph.pkl'}")

In [ ]:
# GRAPH JSON:
import pickle 
print("loading the graph from pickle:")
def load_graph_from_pickle(pickle_path: str):
    """Load a graph from a pickle file."""
    with open(pickle_path, "rb") as f:
        return pickle.load(f)

import json
from pathlib import Path

# convert the graph to a serializable dict
def graph_to_dict(graph):
    return {
        "entities": sorted(list(graph.entities)) if graph.entities else [],
        "edges": sorted(list(graph.edges)) if graph.edges else [],
        "relations": [list(r) for r in sorted(graph.relations)] if graph.relations else [],
        "entity_clusters": {
            k: sorted(list(v)) for k, v in graph.entity_clusters.items()
        } if graph.entity_clusters else {},
        "edge_clusters": {
            k: sorted(list(v)) for k, v in graph.edge_clusters.items()
        } if graph.edge_clusters else {},
        "entity_metadata": {
            k: sorted(list(v)) for k, v in graph.entity_metadata.items()
        } if graph.entity_metadata else {},
    }
#clustered_graph = load_graph_from_pickle(CATEGORY_DIR / f"{CATEGORY}_clustered_graph.pkl")
# save
output_path = CATEGORY_DIR / f"{CATEGORY}_clustered_graph.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(graph_to_dict(clustered_graph), f, ensure_ascii=False, indent=2)

print(f"✅ Graph saved as JSON: {output_path}")

In [ ]:
# ============================================================
# VÉRIFICATION : chaque triplet clusterisé a bien une provenance
# ============================================================

missing = []
empty = []
for triplet in clustered_graph.relations:
    if triplet not in clustered_provenance:
        missing.append(triplet)
    elif not clustered_provenance[triplet]:
        empty.append(triplet)

print(f"Triplets clusterisés         : {len(clustered_graph.relations)}")
print(f"Entrées clustered_provenance : {len(clustered_provenance)}")
print(f"Sans entrée de provenance    : {len(missing)}")
print(f"Avec entrée vide             : {len(empty)}")

if missing:
    print("\n❌ Triplets sans provenance :")
    for t in missing[:10]:
        print(f"    {t}")
else:
    print("\n✅ Tous les triplets clusterisés ont une entrée de provenance.")

# Affichage d'un exemple concret
if clustered_graph.relations:
    example = next(iter(clustered_graph.relations))
    print(f"\n🔎 Exemple : {example}")
    print(f"   Articles d'origine : {clustered_provenance[example]}")


In [ ]:
# 7a. Pickle (fidèle, réutilisable en Python)
prov_out = {k: sorted(v) for k, v in clustered_provenance.items()}
with open(CATEGORY_DIR / "provenance" / f"{CATEGORY}_clustered_provenance.pkl", "wb") as f:
    pickle.dump(prov_out, f)

# 7b. JSON (lisible / interopérable) — on sérialise les triplets en liste
prov_json = [
    {"subject": s, "predicate": p, "object": o, "articles": sorted(arts)}
    for (s, p, o), arts in clustered_provenance.items()
]
with open(CATEGORY_DIR / "provenance" / f"{CATEGORY}_clustered_provenance.json", "w", encoding="utf-8") as f:
    json.dump(prov_json, f, ensure_ascii=False, indent=2)

print(f"\n💾 Provenance sauvegardée :")
print(f"   - {PROVENANCE_DIR / f'{CATEGORY}_clustered_provenance.pkl'}")
print(f"   - {PROVENANCE_DIR / f'{CATEGORY}_clustered_provenance.json'}")

In [ ]:
# ============================================================
# FULL PIPELINE: PROVENANCE PROPAGATION THROUGH CLUSTERING
# Requirements: graphs (dict {i: Graph}), combined_graph, clustered_graph
# ============================================================
from collections import defaultdict
import json
import pickle

# ------------------------------------------------------------
# STEP 1 — Provenance at the raw level (generate + aggregate)
#          100% reliable: each original triplet -> set of articles
# ------------------------------------------------------------

provenance = defaultdict(set)
for i, g in graphs.items():
    article_id = index_to_article_id[i]        # <-- real identifier
    for (subj, rel, obj) in g.relations:
        provenance[(subj, rel, obj)].add(article_id)


print(f"STEP 1 — {len(provenance)} unique triplets (aggregated level)")
assert len(provenance) == len(combined_graph.relations), \
    "⚠️ Inconsistency between raw provenance and combined_graph!"


# ------------------------------------------------------------
# STEP 2 — Propagating provenance to the clustered graph
#          via clustered_graph.relation_clusters
# ------------------------------------------------------------

clustered_provenance = defaultdict(set)
missing_original_triples = []

# relation_clusters : dict[str, list[list[str]]]
#   key   = serialized deduplicated triplet "s\tp\to"
#   value = list of original triplets [ [s,p,o], ... ]
for dedup_key, origin_triples in clustered_graph.relation_clusters.items():
    dedup_triplet = tuple(dedup_key.split("\t"))
    for orig_triple in origin_triples:
        orig_triplet = tuple(orig_triple)
        if orig_triplet in provenance:
            clustered_provenance[dedup_triplet].update(provenance[orig_triplet])
        else:
            missing_original_triples.append(orig_triplet)

# warn about original triplets with unknown provenance
if missing_original_triples:
    print(f"⚠️  {len(missing_original_triples)} original triplet(s) have no known provenance (kept as empty set)")
    for triple in missing_original_triples[:5]:
        print(f"    - {triple}")

# make sure every final triplet has an entry, even empty
for s, p, o in clustered_graph.relations:
    clustered_provenance.setdefault((s, p, o), set())

print(f"STEP 2 — {len(clustered_provenance)} clustered triplets with provenance")


# ------------------------------------------------------------
# STEP 3 — Saving the provenance
#          (tuple keys -> list, to be JSON-serializable)
# ------------------------------------------------------------
# 3a. Pickle (faithful, reusable in Python)
prov_out = {k: sorted(v) for k, v in clustered_provenance.items()}
with open(CATEGORY_DIR / "provenance" / f"{CATEGORY}_clustered_provenance.pkl", "wb") as f:
    pickle.dump(prov_out, f)

# 3b. JSON (readable / interoperable) — triplets serialized as a list
prov_json = [
    {"subject": s, "predicate": p, "object": o, "articles": sorted(arts)}
    for (s, p, o), arts in clustered_provenance.items()
]
with open(CATEGORY_DIR / "provenance" / f"{CATEGORY}_clustered_provenance.json", "w", encoding="utf-8") as f:
    json.dump(prov_json, f, ensure_ascii=False, indent=2)

print(f"\n💾 Provenance saved:")
print(f"   - {CATEGORY_DIR / 'provenance' / f'{CATEGORY}_clustered_provenance.pkl'}")
print(f"   - {CATEGORY_DIR / 'provenance' / f'{CATEGORY}_clustered_provenance.json'}")